# Лабораторная работа 2

Вариант: искусство. X - width_mm, G - десятилетие по acquired

## Задание 1. Подготовка

Подключаем библиотеки и задаём `SEED`, чтобы случайный выбор строк можно было повторить. `numpy` нужен для числовых расчётов, `pandas` — для таблиц, `matplotlib` — для графиков.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

## Задание 2. Загрузка данных

Загружаем CSV в таблицу pandas и проверяем первые строки, типы данных и количество пропусков. Затем создаём столбец `decade` с десятилетием приобретения.

In [ ]:
art = pd.read_csv('records.csv')
art.head(5)

In [ ]:
print(art.dtypes)
print(art['width_mm'].isna().sum())

In [ ]:
decades = []

for acquired_year in art["acquired"]:
    if pd.isna(acquired_year):
        decades.append("Неизвестно")
    else:
        decade = int(acquired_year // 10 * 10)
        decades.append(str(decade))

art["decade"] = decades

`decades = []` создаёт пустой список для нового столбца. Цикл берёт год приобретения каждой работы. `pd.isna()` проверяет пропуск: если год пустой, записываем `Неизвестно`. Иначе операция `acquired_year // 10 * 10` переводит год в десятилетие: например, 1995 превращается в 1990. В конце список добавляется в таблицу как столбец `decade`.

width_mm это float, то есть числа. Пустые ячейки стали пропусками NaN, их 381.

Для G сделала столбец decade: например 1995 -> 1990. Если acquired пустой, то пишу "Неизвестно".

## Задание 3. Пропуски

Выбираем случайные 10% заполненных значений `width_mm`, сохраняем их для проверки и заменяем на `NaN`. Затем восстанавливаем одинаковые пропуски тремя способами: средним, медианой и медианой внутри десятилетия.

In [ ]:
missing_data = art.copy()

removed_rows = missing_data[missing_data['width_mm'].notna()].sample(frac=0.1, random_state=SEED)
removed_index = removed_rows.index
original_widths = removed_rows['width_mm']

missing_data.loc[removed_index, 'width_mm'] = np.nan
print(missing_data['width_mm'].isna().sum())

In [ ]:
mean = missing_data['width_mm'].mean()
median = missing_data['width_mm'].median()

mean_data = missing_data.copy()
mean_data['width_mm'] = mean_data['width_mm'].fillna(mean)

median_data = missing_data.copy()
median_data['width_mm'] = median_data['width_mm'].fillna(median)

decade_data = missing_data.copy()
for decade_value in decade_data['decade'].unique():
    middle_value = decade_data[decade_data['decade'] == decade_value]['width_mm'].median()
    decade_data.loc[(decade_data['decade'] == decade_value) & (decade_data['width_mm'].isna()), 'width_mm'] = middle_value

In [ ]:
error_mean = mean_data.loc[removed_index, 'width_mm'] - original_widths
error_median = median_data.loc[removed_index, 'width_mm'] - original_widths
error_decade = decade_data.loc[removed_index, 'width_mm'] - original_widths

print('среднее: MAE =', np.mean(np.abs(error_mean)), 'MSE =', np.mean(error_mean ** 2))
print('медиана: MAE =', np.mean(np.abs(error_median)), 'MSE =', np.mean(error_median ** 2))
print('медиана по десятилетию: MAE =', np.mean(np.abs(error_decade)), 'MSE =', np.mean(error_decade ** 2))

Лучше всего заполнять медианой по десятилетию, у неё меньше и MAE, и MSE. Ширина работ в разные десятилетия разная (в 1850-х в основном маленькие рисунки, а в 2000-х работы больше).

Медиана лучше среднего по MAE, а среднее лучше по MSE. В данных есть очень широкие работы (до 8920 мм), они тянут среднее вверх (326 против медианы 175). В MSE ошибки возводятся в квадрат, поэтому большие ошибки сильно влияют, а в MAE все ошибки считаются одинаково. Поэтому метрики получаются разные.

## Задание 4. Дубликаты

Выбираем случайные 5% строк и добавляем их точные копии. Затем удаляем повторы по всем столбцам, по `id` и по названию произведения.

In [ ]:
before_copies = art.copy()
copy_rows = before_copies.sample(frac=0.05, random_state=SEED)
with_duplicates = pd.concat([before_copies, copy_rows])

no_duplicates = with_duplicates.copy().drop_duplicates()
unique_id = with_duplicates.copy().drop_duplicates(subset='id')
unique_title = with_duplicates.copy().drop_duplicates(subset='title')

print('до копий:', len(before_copies), before_copies['width_mm'].mean())
print('после копий:', len(with_duplicates), with_duplicates['width_mm'].mean())
print('по всем столбцам:', len(no_duplicates), no_duplicates['width_mm'].mean())
print('по id:', len(unique_id), unique_id['width_mm'].mean())
print('по title:', len(unique_title), unique_title['width_mm'].mean())

In [ ]:
title_ids = no_duplicates.groupby('title')['id'].nunique()
repeated_titles = title_ids[title_ids > 1].index
repeated_rows = no_duplicates[no_duplicates['title'].isin(repeated_titles)].sort_values('title')
print(len(repeated_rows))
repeated_rows[['id', 'title', 'artist', 'width_mm']].head(10)

По всем столбцам и по id удалились только добавленные копии, осталось 8000 строк и среднее как было.

По title удалилось намного больше (осталось 6038), потому что у разных работ бывает одинаковое название, например "[title not known]" или наброски Тёрнера с одинаковыми названиями. У них разные id, то есть это разные работы, и удалять их неправильно.

## Задание 5. Числовые характеристики

Вручную считаем размах, среднее, медиану, дисперсию и стандартное отклонение. После этого сравниваем ручные результаты с NumPy и pandas.

In [ ]:
values = art['width_mm'].dropna().tolist()
amount = len(values)
values.sort()

width_range = values[-1] - values[0]

total_width = 0
for width in values:
    total_width += width
avg_width = total_width / amount

if amount % 2 == 1:
    middle_value = values[amount // 2]
else:
    middle_value = (values[amount // 2 - 1] + values[amount // 2]) / 2

sq_sum = 0
for width in values:
    sq_sum += (width - avg_width) ** 2
variance = sq_sum / amount

std_dev = variance ** 0.5

print(width_range, avg_width, middle_value, variance, std_dev)

In [ ]:
width_data = art['width_mm'].dropna()
print(np.max(width_data) - np.min(width_data), np.mean(width_data), np.median(width_data), np.var(width_data), np.std(width_data))
print(width_data.max() - width_data.min(), width_data.mean(), width_data.median(), width_data.var(), width_data.std())

С NumPy всё совпало. В pandas дисперсия и std немного другие, потому что pandas делит на n-1, а не на n.

## Задание 6. Фильтрация

Выбираем два десятилетия и границу ширины. Сначала оставляем строки, где выполнены оба условия, затем строки, где выполнено хотя бы одно условие.

In [ ]:
both = art[(art['decade'].isin(['1970', '1990'])) & (art['width_mm'] > 500)]
either = art[(art['decade'].isin(['1970', '1990'])) | (art['width_mm'] > 500)]
print(len(both), len(either))

In [ ]:
only_second = either.drop(both.index)
print(len(only_second))
only_second.head(10)

Выбрала десятилетия 1970 и 1990 и границу 500 мм. Во второй таблице строк намного больше, потому что там хватает одного условия: туда попали все работы из 1970 и 1990 (даже узкие) и широкие работы из других десятилетий.

## Задание 7. Порядок операций

Сравниваем два порядка действий:

- сначала отбираем подходящие строки, затем оставляем пять самых широких;
- сначала берём пять самых широких строк всей таблицы, затем применяем фильтр.

Показываем обе таблицы и объясняем, почему результаты могут отличаться.

In [ ]:
top_both = both.sort_values('width_mm', ascending=False).head(5)
top_both

In [ ]:
top_all = art.sort_values('width_mm', ascending=False).head(5)
top_all = top_all[(top_all['decade'].isin(['1970', '1990'])) & (top_all['width_mm'] > 500)]
top_all

Получилось по-разному. В первом случае 5 строк, а во втором только 2, потому что среди 5 самых широких работ всей таблицы три из 2000-х и 2010-х, а из 1970 и 1990 только две. Так что результат не всегда одинаковый, он совпадёт только если топ-5 всей таблицы подходит под условие.

## Задание 8. Группировка

Для каждого десятилетия считаем количество строк, долю от всей таблицы, число пропусков `width_mm`, среднее и медиану. По этим данным строим горизонтальную диаграмму.

In [ ]:
groups = art.groupby('decade')

summary = pd.DataFrame()
summary['count'] = groups.size()
summary['share'] = summary['count'] / len(art)
summary['missing'] = summary['count'] - groups['width_mm'].count()
summary['mean'] = groups['width_mm'].mean()
summary['median'] = groups['width_mm'].median()
summary = summary.sort_values('count', ascending=False)
summary

In [ ]:
chart_counts = summary['count'].sort_values()
plt.barh(chart_counts.index, chart_counts.values)
plt.xlabel('количество')
plt.ylabel('десятилетие')
plt.tight_layout()
plt.show()

Больше всего работ поступило в 1850-х, это больше половины таблицы (почти все в 1856 году). Потом идут 1970, 1990, 1980 и 2000. Меньше всего в 1820, 1830, 1860 и группе "Неизвестно".

Больше всего пропусков ширины в 1880-х (108 из 121).

## Задание 9. Объект и его группа

Добавляем к каждой исходной строке `group_count`, `group_mean` и `delta`. Затем строим график отклонений первых 100 работ и график тех же значений после сортировки по `delta`.

In [ ]:
art['group_count'] = art['decade'].map(summary['count'])
art['group_mean'] = art['decade'].map(summary['mean'])
print(len(art))

art['delta'] = art['width_mm'] - art['group_mean']
art.head(5)

In [ ]:
first_100 = art[art['width_mm'].notna()].head(100)

plt.scatter(range(1, len(first_100) + 1), first_100['delta'])
plt.axhline(0, color='red')
plt.xlabel('номер')
plt.ylabel('delta')
plt.show()

In [ ]:
sorted_delta = first_100.sort_values('delta')

plt.plot(range(1, len(sorted_delta) + 1), sorted_delta['delta'])
plt.xlabel('номер')
plt.ylabel('delta')
plt.show()

В таблице осталось 8000 строк, потому что новые столбцы добавлены к исходному DataFrame, а строки не удалялись. `group_count` одинаков для всех работ одного десятилетия, `group_mean` показывает среднюю ширину этого десятилетия, а `delta` показывает отклонение конкретной работы от среднего. На первом графике точка выше нуля означает ширину выше среднего группы. После сортировки `delta` линия не убывает, потому что значения расположены от меньшего к большему.

## Задание 10. Квантили и boxplot

Считаем квантили 0.25, 0.5 и 0.75. Затем выбираем пять самых больших групп и сравниваем распределение ширины с помощью boxplot.

In [ ]:
art['width_mm'].quantile([0.25, 0.5, 0.75])

In [ ]:
top_decades = summary.head(5).index

box_data = []
for decade_value in top_decades:
    box_data.append(art[art['decade'] == decade_value]['width_mm'].dropna())

plt.boxplot(box_data)
plt.xticks([1, 2, 3, 4, 5], top_decades)
plt.xlabel('десятилетие')
plt.ylabel('ширина, мм')
plt.show()

Медиана выше всего у 2000 и 1970, ниже всего у 1850. Сильнее всего разброс у 2000, у неё самый большой ящик, а у 1850 ящик совсем маленький, там почти все работы небольшие.